# 怎样让智能体通过尝试学会走迷宫？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{
  "learned_success_rate": 1.0,
  "random_success_rate": 0.2222222222222222,
  "learned_mean_steps": 8.705555555555556,
  "learned_mean_return": 9.154999999999998,
  "training_episodes": 1600,
  "evaluation_episodes": 180,
  "elapsed_seconds": 0.229
}


## 1. 补全：终止目标
终点奖励 10，旧Q=2，学习率0.2。终止时不得加下一状态价值，更新结果应为3.6。

In [2]:
old = 2.0
reward = 10.0
alpha = 0.2
done = True
next_value = 100.0
target = reward if done else reward + 0.95 * next_value
updated = old + alpha * (target - old)
assert np.isclose(updated, 3.6)
print(updated)

3.6


## 2. 修改：检查确定性动作
slip=0，从起点向右应到 (0,1)，向上撞墙应留原地，核对奖励。

In [3]:
rng = np.random.default_rng(0)
s, r, done = transition(start, 3, rng, slip=0)
assert s == (0, 1) and np.isclose(r, -0.1) and (not done)
s, r, done = transition(start, 0, rng, slip=0)
assert s == start and np.isclose(r, -0.3)
print('环境边界检查通过')

环境边界检查通过


## 3. 迁移：道路改变后的旧策略
封闭 (0,3)，用旧 Q 表在新图评价，不重新训练；打印性能变化，解释何时需要重新规划或学习。

In [4]:
changed = board.copy()
changed[0, 3] = 1
snapshot = Q.copy()
shifted = evaluate(Q, seed=2026, grid=changed)
assert np.array_equal(Q, snapshot) and shifted.shape == (180, 3)
print('原成功率', learned[:, 0].mean(), '变更后', shifted[:, 0].mean())

原成功率 1.0 变更后 0.022222222222222223
